In [0]:
import pyspark
from pyspark.sql.types import *
import pyspark.sql.functions as f
from pyspark.sql import SparkSession, Row, DataFrame
from pyspark.sql.window import Window
from pyspark.sql.functions import col
from pyspark.sql.functions import when
from typing import Union, Optional, List
from dataclasses import dataclass
from pyspark.sql.types import IntegerType
from functools import reduce
from datetime import timedelta
from pyspark.sql.functions import broadcast
from pyspark.sql.functions import when, lit

#import
#from pls_common_data_store import pls_data_store
#pds = pls_data_store()

#ignore strange depreciation warnings
from warnings import simplefilter 
simplefilter(action='ignore', category=DeprecationWarning)
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")
spark.conf.set("spark.sql.shuffle.partitions","auto")
spark.conf.get("spark.sql.shuffle.partitions")

# spark.conf.set("spark.databricks.queryWatchdog.maxQueryTasks", "50000000")

#### Absolute ROAS / Uplift Automation Built On KPF Closed Loop Dashboard. ![Screenshot 2026-03-18 095335.png](./Screenshot 2026-03-18 095335.png "Screenshot 2026-03-18 095335.png")
- Absolute ROAS and Uplift metrics in Closed Loop 
- Utilized aggregated fuel point redemptions per EHHN per campaign type
- Calculated Adjusted iROAS as: Adjusted Sales Uplift / Total Fuel Points Earned + Campaign Cost
- Calcualted Absolute iROAS as: Adjusted Sales Uplift + Total Fuel Points Earned / Total Points + Campaign Cost
- AROAS follows a similar calculation, but use Adjusted Sales Total instead of Uplift.
- Redemption Cost: Absolute Total Cost - Working Cost (where working cost is campaign cost * multiplier depending on camp type)
- There can be instances of campaigns where the offer is "DOLLAR OFF", rather than Fuel Points. IN this case, instead of using the points detail and target history files to aggregate fuel points per HH / campaign, we can simply refer to media history to obtain redemption cost directly.


In [0]:
# Pre-pivoted closed loop data pulled from closed_loop_campaign_summary notebook
closed_loop_prepivot = spark.read.option("header", "true").csv('abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/closed_loop_summary_tab_INTERMEDIATE.csv')
closed_loop_prepivot = closed_loop_prepivot.filter(f.col('camp_start_date') >= '2023-01-01')
closed_loop_prepivot.display()

# Metadata pulls from KPM
mmci = spark.read.parquet(f'abfss://data@sa8451midsrprd.dfs.core.windows.net/media_meas_campaign_info_v')
mda = spark.read.parquet(f'abfss://measure@sa8451camprd.dfs.core.windows.net/dashboard/campaign/version=v2/source=azure')
points_detail = (spark.read.parquet(f'abfss://data@sa8451kemprd.dfs.core.windows.net/pls_points_v2/'))
mmoi= spark.read.parquet(f'abfss://landingzone@sa8451entlakegrnprd.dfs.core.windows.net/mart/comms/prd/measurement/MEDIA_MEAS_OFFER_INFO')
new_th = spark.read.parquet(f'abfss://landingzone@sa8451entlakegrnprd.dfs.core.windows.net/mart/comms/prd/measurement/TARGET_HISTORY')
old_th = spark.read.parquet(f'abfss://landingzone@sa8451entlakegrnprd.dfs.core.windows.net/mart/comms/prd/measurement/bullseye/TARGET_HISTORY_FULL_20251015/').withColumnRenamed('ehhn', 'hshd_code').select('TARGET_ID', 'HSHD_CODE', 'PRIORITY', 'TEST_CONTROL_ID', 'OFFER_ID', 'DECILE', 'SCORE')
target_history = new_th.union(old_th)
mhtv = spark.read.parquet(f'abfss://data@sa8451midsrprd.dfs.core.windows.net/media_hist_revamped')
redemptions = spark.read.parquet(f'abfss://acds@sa8451posprd.dfs.core.windows.net/transaction_coupon_fct')
downloads = spark.read.parquet(f'abfss://measure@sa8451camprd.dfs.core.windows.net/intermediate/engagements/coupon_downloads/')

####4x REM SSE PUSH XCM iROAS automation

In [0]:
# TO-DO: Only run new campaigns: that is, campaigns that exist in media history that are not inside below path
abs_rem_sse_push_xcm = spark.read.parquet(f'abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/absolute_iroas_rem_sse_push_xcm')
abs_rem_sse_push_xcm.display()

In [0]:
# Handle media history error case. campaign id 138141 XCM EMOD PUSH REM should be type XCM
closed_loop_prepivot = closed_loop_prepivot.withColumn(
    "campaign_type",
    when(f.col("campaign_id") == 138141, lit("XCM")).otherwise(f.col("campaign_type"))
)

closed_loop_prepivot.display()
#closed_loop_prepivot.filter(f.col("campaign_id") == 138141).display()

In [0]:
# only pull new rem sse push
abs_rem_sse_push_xcm = spark.read.parquet('abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/absolute_iroas_rem_sse_push_xcm')
existing_campaign_ids = [row['campaign_id'] for row in abs_rem_sse_push_xcm.select('campaign_id').distinct().collect()]

# These campaigns have dummy UPC or no data in mmoi
exclude_campaign_ids = [79065, 146580, 71365]

closed_loop_prepivot_rem_sse_push = closed_loop_prepivot.filter(
    (f.col("campaign_type") == "XCM") &
    (
        f.col("project_name").contains("XCM SSE PUSH REM") |
        f.col("project_name").contains("XCM EMOD PUSH REM")
    ) &
    (~f.col("project_name").contains("Credit")) &
    (~f.col("project_name").contains("Kroger Pay")) &
    (~f.col("campaign_id").isin(existing_campaign_ids)) &
    (~f.col("campaign_id").isin(exclude_campaign_ids))
)

campaign_ids = [row['campaign_id'] for row in closed_loop_prepivot_rem_sse_push.select('campaign_id').distinct().collect()]
closed_loop_prepivot_rem_sse_push.display()

In [0]:
# Working Cost: Camp Cost * Multiplier (depending on channel type)
mmci_sse_rem_push_working_cost = (mmci.filter(f.col('kpm_duplicated_id').isin(campaign_ids))
    .withColumn('multiplier',
        f.when(f.col('channel') == 'Display Ad', f.lit(0.3520))
         .when(f.col('channel') == 'Email Module', f.lit(0.015))
         .when(f.col('channel') == 'Pandora', f.lit(0.741))
         .when(f.col('channel') == 'Pinterest', f.lit(0.663))
         .when(f.col('channel') == 'Pre-Roll Video', f.lit(.3960))
         .when(f.col('channel') == 'Push Notifications', f.lit(0.038))
         .when(f.col('channel') == 'Roku', f.lit(0.90))
         .when(f.col('channel') == 'Native', f.lit(1))
         .when(f.col('channel') == 'Single Subject Email', f.lit(0.3390))
         .when(f.col('channel') == 'Targeted Digital Coupon', f.lit(0.141))
         .otherwise(f.lit(0))
    )
    .withColumn('working_cost', (f.col('TOT_COST') * f.col('multiplier'))) # Keep as double for now
    .groupBy('KPM_DUPLICATED_ID')
    .agg(
        f.max('CAMP_START_DATE').alias('camp_start_date'),
        f.max('CAMP_END_DATE').alias('camp_end_date'),
        f.sum('TOT_COST').alias('camp_cost_mmci'),
        f.sum('working_cost').alias('working_cost'),
    )
)

mmci_sse_rem_push_working_cost = mmci_sse_rem_push_working_cost.withColumnRenamed('KPM_DUPLICATED_ID', 'campaign_id')
mmci_sse_rem_push_working_cost.display()

In [0]:
mmoi_metadata = (mmoi
    .withColumn('EFFECTIVE_DATE', f.date_format(f.to_date('EFFECTIVE_DATE', 'yyyy-MM-dd'), 'yyyyMMdd'))
    .withColumn('EXPIRATION_DATE', f.date_format(f.to_date('EXPIRATION_DATE', 'yyyy-MM-dd'), 'yyyyMMdd'))
    .withColumn('redemption_barcode', f.lpad(f.col('COUPON_BARCODE').cast('string'), 13, '0'))
    .filter(f.col('KPM_PROJECT_ID').isin(campaign_ids))
    .groupBy('KPM_PROJECT_ID')
    .agg(
        f.collect_set('COUPON_BARCODE').alias('coupon_barcodes'),
        f.collect_set('redemption_barcode').alias('redemption_barcodes'),
        f.first('EFFECTIVE_DATE').alias('effective_date'),
        f.first('EXPIRATION_DATE').alias('expiration_date')
    )
)

mmoi_metadata.display()

In [0]:
# Calculate working cost, iroas (sales uplift / campaign cost), aroas (sales test total / campaign cost)
mhtv_metrics_agg_rem_sse_push = (closed_loop_prepivot_rem_sse_push
    .groupBy('campaign_id')
    .agg(
        f.avg('sales_uplift_total').alias('sales_uplift_total'),
        f.avg('sales_test_total').alias('sales_test_total'),
        f.avg('camp_cost').alias('camp_cost')
    )
    #.withColumn('working_cost', f.col('camp_cost') * 0.3390)
    .withColumn('iroas_original', f.round(f.col('sales_uplift_total') / f.col('camp_cost'), 2))
    .withColumn('aroas_original', f.round(f.col('sales_test_total') / f.col('camp_cost'), 2))
)

mhtv_metrics_agg_rem_sse_push.display()

In [0]:
# Join offer/redemption barcode data with original ROAS + uplift #s with campaign info
mmoi_mhtv_rem_sse_push = (mmoi_metadata
    .join(mhtv_metrics_agg_rem_sse_push, mmoi_metadata["KPM_PROJECT_ID"] == mhtv_metrics_agg_rem_sse_push["campaign_id"], how="right")
)

mmoi_mhtv_rem_sse_push = (mmoi_mhtv_rem_sse_push
    .join(mmci_sse_rem_push_working_cost.select("campaign_id", "working_cost"), on = "campaign_id", how = "inner")
)

mmoi_mhtv_rem_sse_push = (mmoi_mhtv_rem_sse_push
    .join(mmci.select('kpm_project_id', 'target_id', 'project_name'), on = "KPM_PROJECT_ID", how = "inner")
)

mmoi_mhtv_rem_sse_push.display()


In [0]:
target_ids_to_keep = [
    row['target_id'] for row in mmoi_mhtv_rem_sse_push.select('target_id').distinct().collect()
]

# Reduce Target history table with only relevant target ids
test_hhs_slim = (
    target_history
    .filter(f.col('test_control_id') == '1')
    .filter(f.col('target_id').isin(target_ids_to_keep)) 
    .select(f.col('HSHD_CODE').alias('ehhn'), 'target_id')
    .distinct()
)

# Join Target history with metadata
th_filter = (
    test_hhs_slim.join(
        f.broadcast(mmoi_mhtv_rem_sse_push.select('kpm_project_id', 'project_name', 'target_id').distinct()), 
        on='target_id', 
        how='inner'
    )
)

th_filter.display()

In [0]:
# Join target history + metadata with points detail
points_detail = (spark.read.parquet(f'abfss://data@sa8451kemprd.dfs.core.windows.net/pls_points_v2/'))
points_detail_target_history = (broadcast(th_filter)
    .join(points_detail, on='ehhn', how='inner'))
display(points_detail_target_history)

In [0]:
# Select only columns needed from mmoi_mhtv_rem_sse_push
mmoi_mhtv_rem_sse_push_filtered = (mmoi_mhtv_rem_sse_push
    .select(
        'kpm_project_id', 
        'coupon_barcodes', 
        'redemption_barcodes', 
        'effective_date',
        'expiration_date'
    )
)

# Convert transaction date to datetype 
points_detail_target_history = points_detail_target_history.withColumn(
    'trn_dt', f.to_date('trn_dt', 'yyyyMMdd')
)

# Join points detail + target history with dashboard and metadata (iroas, aroas, barcodes, camp info)
points_detail_target_history_offer_info = (
    points_detail_target_history
    .join(
        f.broadcast(mmoi_mhtv_rem_sse_push_filtered),
        on=[
            points_detail_target_history.kpm_project_id == mmoi_mhtv_rem_sse_push_filtered.kpm_project_id,
            (f.array_contains(mmoi_mhtv_rem_sse_push_filtered.coupon_barcodes, points_detail_target_history.offer) | 
             f.array_contains(mmoi_mhtv_rem_sse_push_filtered.redemption_barcodes, points_detail_target_history.offer))
            #points_detail_target_history.trn_dt.between(mmoi_mhtv_mmci_sse_filtered.effective_date, mmoi_mhtv_mmci_sse_filtered.expiration_date)
        ],
        how="inner"
    ).drop(mmoi_mhtv_rem_sse_push_filtered.kpm_project_id)
)

points_detail_target_history_offer_info.display()

In [0]:
# Filter to only transaction date is between effective and expiry date
points_detail_target_history_filtered_offer_info = (points_detail_target_history_offer_info
    .filter(f.col('trn_dt').between('effective_date', 'expiration_date'))
)

points_detail_target_history_filtered_offer_info.display()

In [0]:
#  Sum of points earned across all HHs per campaign
aggregated_df = (points_detail_target_history_filtered_offer_info
    .groupBy('kpm_project_id', 'project_name')
    .agg(
        f.count('ehhn').alias('total_hhs'), 
        f.countDistinct('ehhn').alias('distinct_ehhn'), 
        f.sum('points_earned').alias('total_points_earned')
    )
)

aggregated_df.display()

In [0]:
# Calculations (from Shumaila's code)
final_result_rem_sse_push_df = (mmoi_mhtv_rem_sse_push
    .join(aggregated_df, on=['kpm_project_id', 'project_name'], how='left')
    .withColumn('cost_total_points_earned', f.round(f.col('total_points_earned') * 0.01, 2).cast('double'))
    .withColumn('cost_total_points_redemeed', f.round(f.col('total_points_earned') * 0.016 * 0.5887, 2).cast('double'))
    .withColumn('adj_sales_uplift', f.col("sales_uplift_total").cast('Integer'))
    .withColumn('adj_sales_total', f.col("sales_test_total").cast('Integer'))
    .withColumn('camp_cost', f.round(f.col("camp_cost").cast('double'), 2))
    .withColumn('working_cost', f.round(f.col("working_cost").cast('double'), 2))
    .withColumn('adj_total_cost', f.round((f.col('cost_total_points_earned') + f.col('camp_cost')).cast('double'), 2))
    .withColumn('abs_total_cost', f.round((f.col('cost_total_points_earned') + f.col('working_cost')).cast('double'), 2))
    # Redemption Cost
    .withColumn('redemption_cost', f.col('abs_total_cost') - f.col('working_cost'))
    .withColumn('abs_sales_uplift_earned', f.round((f.col('adj_sales_uplift') - f.col('redemption_cost')).cast('double'), 2))
    .withColumn('abs_sales_test_earned', f.round((f.col('adj_sales_total') - f.col('redemption_cost')).cast('double'), 2))
    .withColumn('adj_iroas', f.round((f.col('adj_sales_uplift') / f.col('adj_total_cost')).cast('double'), 2))
    .withColumn('abs_iroas', f.round((f.col('abs_sales_uplift_earned') / f.col('abs_total_cost')).cast('double'), 2))
    .withColumn('adj_aroas', f.round((f.col('adj_sales_total') / f.col('adj_total_cost')).cast('double'), 2))
    .withColumn('abs_aroas', f.round((f.col('abs_sales_test_earned') / f.col('abs_total_cost')).cast('double'), 2))
)

# Display the final aggregated DataFrame
final_result_rem_sse_push_df.display()

In [0]:
# Dropping array columns (barcodes) to support csv and parquet writing
final_result_rem_sse_push_barcodes_dropped = final_result_rem_sse_push_df.drop('coupon_barcodes', 'redemption_barcodes')
final_result_rem_sse_push_barcodes_dropped.coalesce(1).write.mode("append").parquet('abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/absolute_iroas_rem_sse_push_xcm')

##### REM SSE PUSH TESTING

In [0]:
#SF IDs
'''
xcm_sf_ids = ['SFPRJ1349606', 'SFPRJ1361336', 'SFPRJ1331090', 'SFPRJ1372102', 'SFPRJ1432303', 'SFPRJ1419379']
disp_sf_ids = ['SFPRJ1313773', 'SFPRJ1441818']
mcp_sse_sf_ids = ['SFPRJ1292852', 'SFPRJ1302580', 'SFPRJ1320184', 'SFPRJ1331281', 'SFPRJ1341288', 'SFPRJ1303535', 'SFPRJ1354019', 'SFPRJ1362112', 'SFPRJ1367474', 'SFPRJ1406192', 'SFPRJ1431369', 'SFPRJ1433819']
ol_sse_sf_ids = ['SFPRJ1334280', 'SFPRJ1406195']
rem_sse_push_sf_ids = ['SFPRJ1318288', 'SFPRJ1324975', 'SFPRJ1330990', 'SFPRJ1340725', 'SFPRJ1291631', 'SFPRJ1358041', 'SFPRJ1368696', 'SFPRJ1414741', 'SFPRJ1420017', 'SFPRJ1432022'] 
no_offer_sse_ol = ['SFPRJ1432004']
tdc_sf_ids = ['SFPRJ1317200',	'SFPRJ1317218',	'SFPRJ1317202',	'SFPRJ1317217',	'SFPRJ1317199',	'SFPRJ1319565',	'SFPRJ1319566',	'SFPRJ1319567',	'SFPRJ1322215',	'SFPRJ1322216',	'SFPRJ1322218',	'SFPRJ1330488',	'SFPRJ1330490',	'SFPRJ1331669',	'SFPRJ1331670',	'SFPRJ1331674',	'SFPRJ1333054',	'SFPRJ1333057',	'SFPRJ1333060',	'SFPRJ1335534',	'SFPRJ1335536',	'SFPRJ1340667',	'SFPRJ1340670',	'SFPRJ1340672',	'SFPRJ1342125',	'SFPRJ1342126',	'SFPRJ1342130',	'SFPRJ1342131',	'SFPRJ1342133',	'SFPRJ1342134',	'SFPRJ1342135',	'SFPRJ1342136',	'SFPRJ1342137',	'SFPRJ1342138',	'SFPRJ1350769',	'SFPRJ1350771',	'SFPRJ1350772',	'SFPRJ1355122',	'SFPRJ1355125',	'SFPRJ1355130',	'SFPRJ1355131',	'SFPRJ1357800',	'SFPRJ1357805',	'SFPRJ1357814',	'SFPRJ1357815',	'SFPRJ1366619',	'SFPRJ1366622',	'SFPRJ1366623',	'SFPRJ1366629',	'SFPRJ1366632',	'SFPRJ1368123',	'SFPRJ1368128',	'SFPRJ1368131',	'SFPRJ1372098',	'SFPRJ1372099',	'SFPRJ1372100',	'SFPRJ1372101',	'SFPRJ1372103',	'SFPRJ1372106',	'SFPRJ1372112',	'SFPRJ1403880',	'SFPRJ1410311',	'SFPRJ1410314',	'SFPRJ1413968',	'SFPRJ1413969',	'SFPRJ1413970',	'SFPRJ1413971',	'SFPRJ1413973', 'SFPRJ1420546',	'SFPRJ1420548',	'SFPRJ1420549',	'SFPRJ1420696',	'SFPRJ1420707',	'SFPRJ1420697',	'SFPRJ1420703',	'SFPRJ1432301',	'SFPRJ1432308',	'SFPRJ1432311',	'SFPRJ1432313',	'SFPRJ1432314',	'SFPRJ1432306',	'SFPRJ1432309',	'SFPRJ1432315',	'SFPRJ1441205',	'SFPRJ1441210',	'SFPRJ1441212',	'SFPRJ1445313'] 
ol_tdc_sf_ids = ['SFPRJ1343831', 'SFPRJ1413976', 'SFPRJ1415936']

#pulling campaign ids
xcm_camp_ids = mmci.filter(f.col('SALESFORCE_ID').isin(xcm_sf_ids)).select('kpm_duplicated_id').distinct().rdd.flatMap(lambda x: x).collect()
disp_camp_ids = mmci.filter(f.col('SALESFORCE_ID').isin(disp_sf_ids)).select('kpm_duplicated_id').distinct().rdd.flatMap(lambda x: x).collect()
mcp_sse_camp_ids = mmci.filter(f.col('SALESFORCE_ID').isin(mcp_sse_sf_ids)).select('kpm_duplicated_id').distinct().rdd.flatMap(lambda x: x).collect()
ol_mcp_sse_camp_ids = mmci.filter(f.col('SALESFORCE_ID').isin(ol_sse_sf_ids)).select('kpm_duplicated_id').distinct().rdd.flatMap(lambda x: x).collect()
rem_sse_push_camp_ids = mmci.filter(f.col('SALESFORCE_ID').isin(rem_sse_push_sf_ids)).select('kpm_duplicated_id').distinct().rdd.flatMap(lambda x: x).collect()
tdc_camp_ids = mmci.filter(f.col('SALESFORCE_ID').isin(tdc_sf_ids)).select('kpm_duplicated_id').distinct().rdd.flatMap(lambda x: x).collect()
ol_tdc_camp_ids = mmci.filter(f.col('SALESFORCE_ID').isin(ol_tdc_sf_ids)).select('kpm_duplicated_id').distinct().rdd.flatMap(lambda x: x).collect()
no_offer_sse_ol_camp_ids = mmci.filter(f.col('SALESFORCE_ID').isin(no_offer_sse_ol)).select('kpm_duplicated_id').distinct().rdd.flatMap(lambda x: x).collect()

print('xcm_camp_ids:', xcm_camp_ids)
print('disp_camp_ids:', disp_camp_ids)
print('mcp_sse_camp_ids:', mcp_sse_camp_ids)
print('ol_mcp_sse_camp_ids:', ol_mcp_sse_camp_ids)
print('rem_sse_push_camp_ids:', rem_sse_push_camp_ids)
print('tdc_camp_ids:', tdc_camp_ids)
print('ol_tdc_camp_ids:', ol_tdc_camp_ids)
'''

In [0]:
'''
camp_ids = rem_sse_push_camp_ids

results = []
for camp_id in camp_ids:
    mmci_filter = mmci.filter(f.col('kpm_project_id') == camp_id)
    campaign_ref = mmci_filter.select('KPM_PROJECT_ID', 'project_name').distinct()
    mmci_filter = (mmci.filter(f.col('kpm_duplicated_id') == camp_id)
               .withColumn('CAMP_START_DATE', f.to_date('CAMP_START_DATE', 'yyyy-MM-dd'))
               .withColumn('CAMP_END_DATE', f.to_date('CAMP_END_DATE', 'yyyy-MM-dd'))
               .select('KPM_DUPLICATED_ID', 'KPM_PROJECT_ID', 'CHANNEL', 'CAMP_START_DATE', 'CAMP_END_DATE', 'PROJECT_NAME', 'TOT_COST', 'TARGET_ID')
               .withColumn('multiplier',
                           f.when(f.col('channel') == 'Display Ad', f.lit(0.3520))
                           .when(f.col('channel') == 'Email Module', f.lit(0.015))
                           .when(f.col('channel') == 'Pandora', f.lit(0.741))
                           .when(f.col('channel') == 'Pinterest', f.lit(0.663))
                           .when(f.col('channel') == 'Pre-Roll Video', f.lit(.3960))
                           .when(f.col('channel') == 'Push Notifications', f.lit(0.038))
                           .when(f.col('channel') == 'Roku', f.lit(0.90))
                           .when(f.col('channel') == 'Native', f.lit(1))
                           .when(f.col('channel') == 'Single Subject Email', f.lit(0.3390))
                           .when(f.col('channel') == 'Targeted Digital Coupon', f.lit(0.141))
                           .otherwise(f.lit(0)))
               .withColumn('working_cost', (f.col('TOT_COST') * f.col('multiplier')).cast('integer')))
    
    mmoi_filter = (mmoi
        .filter(f.col('kpm_project_id') == camp_id)
        .withColumn('DISPLAY_START_DATE', f.to_date('DISPLAY_START_DATE', 'yyyy-MM-dd'))
        .withColumn('DISPLAY_END_DATE', f.to_date('DISPLAY_END_DATE', 'yyyy-MM-dd'))
        .withColumn('EFFECTIVE_DATE', f.date_format('EFFECTIVE_DATE', 'yyyyMMdd'))
        .withColumn('EXPIRATION_DATE', f.date_format('EXPIRATION_DATE', 'yyyyMMdd'))
        .select('OFFER_ID', 'KPM_PROJECT_ID', 'DISPLAY_START_DATE', 'DISPLAY_END_DATE', 'EFFECTIVE_DATE', 'EXPIRATION_DATE', 'COUPON_BARCODE'))
    
    # MMOI BARCODE VERIFICATION-------------------------------
    mmoi_filter.display()
    # --------------------------------------------------------

    mhtv_filtered = mhtv.filter((f.col('campaign_id').isin(camp_id)) & (f.col('adjusted_top_performer') == 'Top-Performer_KRO') & (f.col('modality') == 'All Modalities'))
    mhtv_filtered_agg = mhtv_filtered.groupBy('campaign_id').agg(f.sum('sales_uplift_total').alias('sales_uplift_total'),f.sum('sales_test_total').alias('sales_test_total'), f.sum('camp_cost').alias('camp_cost')).withColumn('working_cost', f.col('camp_cost') * 0.3390).withColumn('iroas', f.round(f.col('sales_uplift_total') / f.col('camp_cost'), 2)).withColumn('aroas', f.round(f.col('sales_test_total') / f.col('camp_cost'), 2))


    #  iroas, sales uplift, sales test, working cost verification -------------------------------
    mhtv_filtered_agg.display()
    # --------------------------------------------------------

    target_id = mmci_filter.select('target_id').distinct().first()['target_id']
    coupon_barcodes = mmoi_filter.select('coupon_barcode').distinct().rdd.flatMap(lambda x: x).collect()
    redemption_barcodes = (mmoi_filter.select('coupon_barcode').distinct()
                           .withColumn('redemption_barcode', f.lpad(f.col('coupon_barcode').cast('string'), 13, '0'))
                           .select('redemption_barcode').distinct().rdd.flatMap(lambda x: x).collect())
    effective_date = mmoi_filter.select('EFFECTIVE_DATE').distinct().first()['EFFECTIVE_DATE']
    expiration_date = mmoi_filter.select('EXPIRATION_DATE').distinct().first()['EXPIRATION_DATE']
    sales_uplift_total_original = mhtv_filtered_agg.select('sales_uplift_total').first()['sales_uplift_total']
    sales_test_total_original = mhtv_filtered_agg.select('sales_test_total').first()['sales_test_total']
    iroas_original = mhtv_filtered_agg.select('iroas').first()['iroas']
    aroas_original = mhtv_filtered_agg.select('aroas').first()['aroas']
    camp_cost = mhtv_filtered_agg.select('camp_cost').first()['camp_cost']
    working_cost = mmci_filter.groupBy('kpm_duplicated_id').agg(f.sum('working_cost').alias('working_cost')).select('working_cost').first()['working_cost']

    print('camp_id =', camp_id)

    th_filter = target_history.filter(f.col('target_id') == target_id).filter(f.col('test_control_id') == '1').withColumnRenamed('HSHD_CODE', 'ehhn').select('ehhn').distinct().withColumn('target_id', f.lit(target_id))

    points_detail = spark.read.parquet(f'abfss://data@sa8451kemprd.dfs.core.windows.net/pls_points_v2/').filter(f.col('trn_dt').between(effective_date, expiration_date))
    points_detail_th = points_detail.filter((f.col('offer').isin(coupon_barcodes)) | (f.col('offer').isin(redemption_barcodes))).join(th_filter, on='ehhn', how='inner').join(mmci_filter, on='target_id', how='inner')

    aggregated_df = points_detail_th.groupBy('KPM_PROJECT_ID', 'project_name').agg(f.count('ehhn').alias('total_hhs'), f.countDistinct('ehhn').alias('distinct_ehhn'), f.sum('points_earned').alias('total_points_earned'))

    # VALIDATE POINTS EARNED -------------------------------------------------------
    aggregated_df.display()
    # --------------------------------------------------------

    agg_zero_points = campaign_ref.join(aggregated_df, on = ['KPM_PROJECT_ID', 'project_name'], how = 'left').withColumn('total_hhs', f.when(f.col('total_hhs').isNull(), 0).otherwise(f.col('total_hhs'))).withColumn('distinct_ehhn', f.when(f.col('distinct_ehhn').isNull(), 0).otherwise(f.col('distinct_ehhn'))).withColumn('total_points_earned', f.when(f.col('total_points_earned').isNull(), 0).otherwise(f.col('total_points_earned')))

    result_df = (agg_zero_points
                 .withColumn('cost_total_points_earned', f.round(f.col('total_points_earned') * 0.01, 2).cast('double'))
                 .withColumn('cost_total_points_redemeed', f.round(f.col('total_points_earned') * 0.016 * 0.5887, 2).cast('double'))
                 .withColumn('adj_sales_uplift', f.lit(sales_uplift_total_original).cast('Integer'))
                 .withColumn('adj_sales_total', f.lit(sales_test_total_original).cast('Integer'))
                 .withColumn('camp_cost', f.round(f.lit(camp_cost).cast('double'), 2))
                 .withColumn('working_cost', f.round(f.lit(working_cost).cast('double'), 2))
                 .withColumn('abs_sales_uplift_earned', f.round((f.col('adj_sales_uplift') - f.col('cost_total_points_earned')).cast('double'), 2))
                 .withColumn('abs_sales_test_earned', f.round((f.col('adj_sales_total') - f.col('cost_total_points_earned')).cast('double'), 2))
                 .withColumn('adj_total_cost', f.round((f.col('cost_total_points_earned') + f.col('camp_cost')).cast('double'), 2))
                 .withColumn('abs_total_cost', f.round((f.col('cost_total_points_earned') + f.col('working_cost')).cast('double'), 2))
                 .withColumn('adj_iroas', f.round((f.col('adj_sales_uplift') / f.col('adj_total_cost')).cast('double'), 2))
                 .withColumn('abs_iroas', f.round((f.col('abs_sales_uplift_earned') / f.col('abs_total_cost')).cast('double'), 2))
                 .withColumn('adj_aroas', f.round((f.col('adj_sales_total') / f.col('adj_total_cost')).cast('double'), 2))
                 .withColumn('abs_aroas', f.round((f.col('abs_sales_test_earned') / f.col('abs_total_cost')).cast('double'), 2)))

    results.append(result_df)

final_result_df_sse = reduce(lambda df1, df2: df1.union(df2), results)
'''

#### OFFSITE XCM + DISPLAY iROAS automation
- When XCM barcodes are added to master tracker after measurement comes in, input manually and run


In [0]:
abs_offsite_xcm = spark.read.parquet(f'abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/absolute_iroas_xcm_disp')
abs_offsite_xcm.display()

In [0]:
# filter current dashboard to only include XCM
closed_loop_prepivot_xcm_disp = closed_loop_prepivot.filter(
    (f.col("campaign_type") == "DISPLAY_AD") | (f.col("campaign_type") == "XCM") | (f.col("campaign_type") == "EMOD")
)
closed_loop_prepivot_xcm_disp.display()

In [0]:
# Replace this with read-in Excel file containing coupon codes, campaign IDs, and campaign type

data = [ # 2025 Spring Easter Mothers Day
    #(800000538427, 131511, 'XCM'),
    #(800000538426, 131511, 'XCM'),
    #(800000538459, 131511, 'XCM'),
    #(800000538460, 131511, 'XCM'),
    #(800000727746, 131511, 'XCM'),
    #(800000727744, 131511, 'XCM'),
    #(800000727743, 131511, 'XCM'),
    #(800000727745, 131511, 'XCM'),
    #(800000786477, 131511, 'XCM'),
    #(800000786476, 131511, 'XCM'),
    #(800000786479, 131511, 'XCM'),
    #(800000786478, 131511, 'XCM'), ]

    #(800000176164, 153839, 'XCM'),
    #(800000175826, 153839, 'XCM'),
    #(800000176170, 153839, 'XCM'),
    #(800000176075, 153839, 'XCM'),
    #(800000176162, 153839, 'XCM'),
    #(800000176161, 153839, 'XCM'),
    #(800000175835, 153839, 'XCM'),
    #(800000176087, 153839, 'XCM'),
]
'''
    # 2025 HOLIDAY XCM
    (800000176164, 153839, 'XCM'),
    (800000175826, 153839, 'XCM'),
    (800000176170, 153839, 'XCM'),
    (800000176075, 153839, 'XCM'),
    (800000176162, 153839, 'XCM'),
    (800000176161, 153839, 'XCM'),
    (800000175835, 153839, 'XCM'),
    (800000176087, 153839, 'XCM'),
    # 2025 WInter 4x DISP
    (800000192179, 161613, 'DISPLAY_AD'),
    (800000192181, 161613, 'DISPLAY_AD'),
    (800000192180, 161613, 'DISPLAY_AD'),
    (800000192838, 161613, 'DISPLAY_AD'),

    # Campaigns w No Offer
    (000000000000, 147400, 'XCM'),
    (000000000000, 136421, 'XCM'),
    # 2025 Fathers Day Summer
    (800000835408, 139466, 'XCM'),
    (800000835410, 139466, 'XCM'),
    (800000831624, 139466, 'XCM'),
    (800000835409, 139466, 'XCM'),
    (800000132655, 139466, 'XCM'),
    (800000132654, 139466, 'XCM'),
    (800000132656, 139466, 'XCM'),
    (800000131272, 139466, 'XCM'),
    # 2025 BTS
    (800000138292, 141800, 'XCM'),
    (800000138286, 141800, 'XCM'),
    (800000138299, 141800, 'XCM'),
    (800000138300, 141800, 'XCM'),
    # 2025 Valentines Day
    (800000531437, 128642, 'DISPLAY_AD'),
    (800000531431, 128642, 'DISPLAY_AD'),
    (800000531433, 128642, 'DISPLAY_AD'),
    (800000531432, 128642, 'DISPLAY_AD'),
    # 2025 Halloween
    (800000164400, 155120, 'DISPLAY_AD'), 
    (800000164397, 155120, 'DISPLAY_AD'), 
    (800000164398, 155120, 'DISPLAY_AD'), 
    (800000164399, 155120, 'DISPLAY_AD'),
    # 2025 Fall 4X Dinner Game
    (800000149690, 151089, 'XCM'),
    (800000149688, 151089, 'XCM'),
    (800000149325, 151089, 'XCM'),
    (800000149652, 151089, 'XCM'),
    (800000150177, 151089, 'XCM'),
    (800000150172, 151089, 'XCM'),
    (800000150183, 151089, 'XCM'),
    (800000150201, 151089, 'XCM'),
    # 2024 Easter Q1
    (800000283555, 85399, 'XCM'),
    (800000283554, 85399, 'XCM'),
    # 2024 BTS Q2
    (800000597889, 103442, 'XCM'),
    (800000597689, 103442, 'XCM'),
    (800000619889, 103442, 'XCM'),
    (800000620289, 103442, 'XCM'),
    # 2024 Holiday Q3
    (800000616531, 116942, 'XCM'),
    (800000616534, 116942, 'XCM'),
    (800000616535, 116942, 'XCM'),
    (800000616530, 116942, 'XCM'),
    # 2024 Mothers Day Fathers Day
    (800000320450, 91440, 'XCM'),
    (800000320449, 91440, 'XCM'),
    (800000380740, 91440, 'XCM'),
    (800000380739, 91440, 'XCM'),
    (800000364723, 91440, 'XCM'),
    (800000365249, 91440, 'XCM'),
'''  


schema = ["coupon_barcode", "campaign_id", "campaign_type"]
coupon_df = spark.createDataFrame(data, schema)
coupon_df.display()

campaign_ids = coupon_df.select('campaign_id').distinct().rdd.flatMap(lambda x: x).collect()
print(campaign_ids)

In [0]:
# Working Cost: Camp Cost * Multiplier (depending on channel type)
mmci_xcm_display_working_cost = (mmci.filter(f.col('kpm_duplicated_id').isin(campaign_ids))
    .withColumn('multiplier',
        f.when(f.col('channel') == 'Display Ad', f.lit(0.3520))
         .when(f.col('channel') == 'Email Module', f.lit(0.015))
         .when(f.col('channel') == 'Pandora', f.lit(0.741))
         .when(f.col('channel') == 'Pinterest', f.lit(0.663))
         .when(f.col('channel') == 'Pre-Roll Video', f.lit(.3960))
         .when(f.col('channel') == 'Push Notifications', f.lit(0.038))
         .when(f.col('channel') == 'Roku', f.lit(0.90))
         .when(f.col('channel') == 'Native', f.lit(1))
         .when(f.col('channel') == 'Single Subject Email', f.lit(0.3390))
         .when(f.col('channel') == 'Targeted Digital Coupon', f.lit(0.141))
         .otherwise(f.lit(0))
    )
    .withColumn('working_cost', (f.col('TOT_COST') * f.col('multiplier'))) # Keep as double for now
    .groupBy('KPM_DUPLICATED_ID')
    .agg(
        f.max('CAMP_START_DATE').alias('camp_start_date'),
        f.max('CAMP_END_DATE').alias('camp_end_date'),
        f.sum('TOT_COST').alias('camp_cost_mmci'),
        f.sum('working_cost').alias('working_cost'),
    )
)

mmci_xcm_display_working_cost = mmci_xcm_display_working_cost.withColumnRenamed('KPM_DUPLICATED_ID', 'campaign_id')
mmci_xcm_display_working_cost.display()

In [0]:
# Pulling coupons from Excel / Temp DF
coupon_barcodes_agg = (coupon_df
    .withColumn('coupon_barcode', f.col('coupon_barcode').cast('string'))
    .withColumn('redemption_barcode', f.lpad(f.col('coupon_barcode'), 13, '0'))
    .groupBy('campaign_id').agg(
        f.collect_set('coupon_barcode').alias('coupon_barcodes'),
        f.collect_set('redemption_barcode').alias('redemption_barcodes')
    )
    .withColumnRenamed('campaign_id', 'kpm_duplicated_id')
)

mmci_xcm_display_working_cost_barcodes = coupon_barcodes_agg.join(
    mmci_xcm_display_working_cost.withColumnRenamed('campaign_id', 'kpm_duplicated_id'),
    'kpm_duplicated_id', 'inner'
)
display(mmci_xcm_display_working_cost_barcodes)

In [0]:
measurement_path = 'abfss://measure@sa8451camprd.dfs.core.windows.net'
version = 'v2'
source = 'azure'

# Get all job ids for xcm and display ad campaigns (using campaign id) and make into a list
job_id_rows = (mda.filter(f.col('campaign_id').isin(campaign_ids))
               .filter(f.col('downstream_ready') == 'true')
               .select('campaign_id', 'job_id', 'campaign_type').distinct().collect())

# Create a dictionary where the key is campaign_id and the value is a tuple containing both job_id and campaign_type (Need to fetch campaign type to use in path call)
camp_id_job_type_dict = {
    row['campaign_id']: {'job_id': row['job_id'], 'camp_type': row['campaign_type']} 
    for row in job_id_rows
}

print(camp_id_job_type_dict)

In [0]:
# Loop through key value pair of kpm id + job id (to use in projection path dynamically), loop needed here i think because paths are different per kpm id 
projection_results = []
for camp_id, camp_info in camp_id_job_type_dict.items():
  job_id = camp_info['job_id']
  camp_type = camp_info['camp_type'] 

  # Filters different depending on campaign type - XCM (filters are shumailas logic)
  if camp_type == "XCM":
    print(camp_id, camp_type, job_id)
    projection = (spark.read.parquet(
      f'{measurement_path}/reports/projection/version={version}/source={source}/campaign_type={camp_type}/campaign_id={camp_id}/job_id={job_id}')
        .filter(~f.col('product_group').like('%_online%'))
        .filter(~f.col('product_group').like('%_in_store%'))
        .filter(f.col('segment') == 'ALL')
        .withColumn('scale_factor', f.round(f.col('projection_factor').cast('double'), 2))
        .withColumn('kpm_duplicated_id', f.lit(camp_id))
        .select("kpm_duplicated_id", "scale_factor", "projection_factor")
    )
    projection_results.append(projection)

  # Filters different depending on campaign type - DISPLAY (filters are Shumailas logic) or EMOD (?)
  elif camp_type != "XCM":
    print(camp_id, camp_type, job_id)
    projection = (spark.read.parquet(
      f'{measurement_path}/reports/media_metrics/campaign/version={version}/source={source}/campaign_type={camp_type}/campaign_id={camp_id}/job_id={job_id}')
        .filter(f.col('modality') == 'All Modalities')
        .filter(f.col('segment') == 'ALL')
        .filter(f.col('sub_segment') == 'ALL')
        #.filter(f.col('product_group') == 'Third Party And Open Loop 010225')
        .filter(f.col('rom') == 'Kroger Only').filter(f.col('metric') == 'scale_factor')
        .withColumnRenamed('value', 'projection_factor').select('projection_factor').distinct()
        .withColumn('scale_factor', f.round(f.col('projection_factor').cast('double'), 2))
        .withColumn("kpm_duplicated_id", f.lit(camp_id))
        .select("kpm_duplicated_id", "scale_factor", "projection_factor")
    )
    projection_results.append(projection)

# Union All Data to obtain scale factor + projection factor per kpm duplicated id
projection_all = reduce(lambda df1, df2: df1.union(df2), projection_results).dropDuplicates()

display(projection_all)

In [0]:
# Get Test HHs per campaign, along with # Distinct Test Households for the campaign

hhs_cont_test_list = [] 
distinct_test_hhs_list = [] 

for camp_id, camp_info in camp_id_job_type_dict.items():
    job_id = camp_info['job_id']
    camp_type = camp_info['camp_type'] 
    print(job_id, camp_type)

    # HH matching based on camp_type XCM
    if camp_type == "XCM":
        hhs_match_original = spark.read.parquet(
            f'{measurement_path}/intermediate/hh_matching/match/campaign_type={camp_type}/campaign_id={camp_id}/job_id={job_id}'
        ).withColumn('kpm_duplicated_id', f.lit(camp_id))
        hh_cont = hhs_match_original.select('kpm_duplicated_id', f.col('cont_hh').alias('ehhn')).withColumn('hhgroup', f.lit('CONT'))
        hh_test = hhs_match_original.select('kpm_duplicated_id', f.col('test_hh').alias('ehhn')).withColumn('hhgroup', f.lit('TEST'))

        hh_cont_test = hh_cont.union(hh_test).withColumn('kpm_duplicated_id', f.lit(camp_id))
        distinct_test_hh = hh_cont_test.filter(f.col('hhgroup') == 'TEST').groupBy('hhgroup', 'kpm_duplicated_id').agg(f.countDistinct('ehhn').alias('distinct_test_hhs'))

        hhs_cont_test_list.append(hh_cont_test)
        distinct_test_hhs_list.append(distinct_test_hh)


    # HH matching based on camp_type DISPLAY_AD (Change this for other Channel Types)
    elif camp_type == "DISPLAY_AD":
        hhs_match_original = spark.read.parquet(
            #f'abfss://measure@sa8451camprd.dfs.core.windows.net/inputs/households/campaign_type={camp_type}/campaign_id={camp_id}/job_id={job_id}'
            f'abfss://measure@sa8451camprd.dfs.core.windows.net/inputs/households/campaign_type=DISPLAY_AD/campaign_id=128642/job_id=2025-05-12-1747078257-MAEKM'
        ).withColumn('kpm_duplicated_id', f.lit(camp_id))
        hhs_cont = hhs_match_original.filter(f.col("hhgroup") == "CONT").select('kpm_duplicated_id', 'hshd_code', 'hhgroup').withColumnRenamed('hshd_code', 'ehhn')
        hhs_test = hhs_match_original.filter(f.col("hhgroup") == "TEST").select('kpm_duplicated_id', 'hshd_code', 'hhgroup').withColumnRenamed('hshd_code', 'ehhn')

        hh_cont_test = hhs_cont.union(hhs_test).withColumn('kpm_duplicated_id', f.lit(camp_id))
        distinct_test_hh = hh_cont_test.filter(f.col('hhgroup') == 'TEST').groupBy('hhgroup', 'kpm_duplicated_id').agg(f.countDistinct('ehhn').alias('distinct_test_hhs'))

        hhs_cont_test_list.append(hh_cont_test)
        distinct_test_hhs_list.append(distinct_test_hh)

if hhs_cont_test_list:
    hhs_cont_test = reduce(lambda df1, df2: df1.unionByName(df2), hhs_cont_test_list)
    hhs_cont_test.persist()
    
if distinct_test_hhs_list:
    distinct_test_hhs = reduce(lambda df1, df2: df1.unionByName(df2), distinct_test_hhs_list)

distinct_test_hhs = distinct_test_hhs.dropDuplicates(['kpm_duplicated_id', 'hhgroup']) 
distinct_test_hhs.display()

In [0]:
# Join points detail with households
points_detail = (spark.read.parquet(f'abfss://data@sa8451kemprd.dfs.core.windows.net/pls_points_v2/'))
points_detail_ehhns = (points_detail
    .join(broadcast(hhs_cont_test), on='ehhn', how='inner'))

display(points_detail_ehhns)
# Making sure all campaign_ids have HH points detail
distinct_kpm_ids = points_detail_ehhns.select('kpm_duplicated_id').distinct()
display(distinct_kpm_ids)

In [0]:
# Join Points Detail w/ Dashboard and Select only coupon and redemption barcode offers
points_detail_ehhns_barcodes = (points_detail_ehhns
    .withColumn('trn_dt', f.to_date('trn_dt', 'yyyyMMdd'))
    .join(mmci_xcm_display_working_cost_barcodes.select('kpm_duplicated_id', 'coupon_barcodes', 'redemption_barcodes', 'camp_start_date', 'camp_end_date'), 
          on = "kpm_duplicated_id",
          how='inner')
    .filter(
        f.array_contains(f.col('coupon_barcodes'), f.col('offer')) |
        f.array_contains(f.col('redemption_barcodes'), f.col('offer'))
    )
)

# Filter transaction date within camp_start and camp_end date
points_detail_ehhn_barcodes_filtered = (points_detail_ehhns_barcodes
    .filter(
        (f.col('trn_dt') >= f.to_date(f.col('camp_start_date'), 'yyyyMMdd')) &
        (f.col('trn_dt') <= f.to_date(f.col('camp_end_date'), 'yyyyMMdd'))
    )
)

points_detail_ehhn_barcodes_filtered.display()

In [0]:
# Calculate aggregate points detail (earned and redeemed) CONTROL: total_hhs,	distinct_hhs, total_points_earned, total_points_redeemed, 
# distinct_test_hhs. total_measured_hhs	earn_per_hh	measured_hhs_earned
points_detail_ehhn_barcodes_filtered_CONT = (points_detail_ehhn_barcodes_filtered
    .filter(f.col('hhgroup') == 'CONT')
    .groupBy('hhgroup', 'kpm_duplicated_id')
    .agg(
      f.count('ehhn').alias('total_hhs'),
      f.countDistinct('ehhn').alias('distinct_hhs'), 
      f.sum('points_earned').alias('total_points_earned'), 
      f.sum('points_redeemed').alias('total_points_redeemed')
    )
    .join(
      distinct_test_hhs.select('kpm_duplicated_id', 'distinct_test_hhs'), on = 'kpm_duplicated_id', how = 'inner'
    )
    .join(
      coupon_df.select(col("campaign_id").alias("kpm_duplicated_id"), "campaign_type"), on = "kpm_duplicated_id", how = "left"
    )
    .withColumn(
      'total_measured_hhs', f.col('distinct_test_hhs')
    )
    # Shumaila's logic - denominator is different depending on camp_type
    .withColumn(
      'earn_per_hh', when(f.col("campaign_type") == "XCM", f.col('total_points_earned') / f.col('total_measured_hhs')
      ).otherwise(
        f.col('total_points_earned') / f.col('total_measured_hhs')
      )
    )
    .withColumn('measured_hhs_earned', f.col('earn_per_hh') * f.col('total_measured_hhs'))
)

points_detail_ehhn_barcodes_filtered_CONT.display()

In [0]:
# Calculate aggregate points detail (earned and redeemed) TEST: total_hhs,	distinct_hhs, total_points_earned, total_points_redeemed, 
# distinct_test_hhs. total_measured_hhs	earn_per_hh	measured_hhs_earned
points_detail_ehhn_barcodes_filtered_TEST = (points_detail_ehhn_barcodes_filtered
    .filter(f.col('hhgroup') == 'TEST')
    .groupBy('hhgroup', 'kpm_duplicated_id')
    .agg(
      f.count('ehhn').alias('total_hhs'),
      f.countDistinct('ehhn').alias('distinct_hhs'), 
      f.sum('points_earned').alias('total_points_earned'), 
      f.sum('points_redeemed').alias('total_points_redeemed')
    )
    .join(
      distinct_test_hhs.select('kpm_duplicated_id', 'distinct_test_hhs'), on = 'kpm_duplicated_id', how = 'inner'
    )
    .join(
      coupon_df.select(col("campaign_id").alias("kpm_duplicated_id"), "campaign_type"), on = "kpm_duplicated_id", how = "left"
    )
    .withColumn(
      'total_measured_hhs', f.col('distinct_test_hhs')
    )

    # Shumaila's logic - denominator is different depending on camp_type
    .withColumn(
      'earn_per_hh', when(f.col("campaign_type") == "XCM", f.col('total_points_earned') / f.col('total_measured_hhs')
      # Fixed on 3-2 to make it total points / total measured hhs regardless of tactic                    
      ).otherwise(
        f.col('total_points_earned') / f.col('total_measured_hhs')
      )
    )
    .withColumn('measured_hhs_earned', f.col('earn_per_hh') * f.col('total_measured_hhs'))
)

points_detail_ehhn_barcodes_filtered_TEST.display()


In [0]:
# UNION of test and control hh and points numbers
spark.conf.set("spark.sql.shuffle.partitions", "2000") 
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.shuffle.spill", "true")

points_detail_final = points_detail_ehhn_barcodes_filtered_CONT.union(points_detail_ehhn_barcodes_filtered_TEST)
points_detail_final.display()

In [0]:
# Preprocessing for efficient cluster performance 
spark.conf.set("spark.sql.shuffle.partitions", "1024")
spark.conf.set("spark.sql.adaptive.enabled", "true")
hhgroups = ['TEST', 'CONT']

# Pivot to have control and test HH and Points #s on new columns
points_detail_final_pivoted_cont_test = (
    points_detail_final
        .groupBy('kpm_duplicated_id')
        .pivot('hhgroup', hhgroups)
        .agg(
            f.first('total_hhs').alias('total_original_hhs'),
            f.first('distinct_hhs').alias('distinct_original_hhs'),
            f.first('total_measured_hhs').alias('total_measured_hhs'),
            f.first('total_points_earned').alias('total_points_earned'),
            f.first('measured_hhs_earned').alias('measured_hhs_earned'),
        )
)

# Add scale factor that we previously calculated - we need this for "inc_total_points_earned_scaled" which is also used for other calcs 
# Bring in campaign cost and working cost that was previously calculated
# Bring in sales uplift and sales test from kpf dashboard
points_detail_final_pivoted_cont_test = (points_detail_final_pivoted_cont_test
    .join(f.broadcast(projection_all), on='kpm_duplicated_id', how='inner')
    .join(f.broadcast(mmci_xcm_display_working_cost_barcodes.select('kpm_duplicated_id','camp_cost_mmci', 'working_cost')), on='kpm_duplicated_id', how='inner')
    .join(f.broadcast(closed_loop_prepivot.select('kpm_duplicated_id', 'sales_uplift_total', 'sales_test_total')), on='kpm_duplicated_id', how='inner')
)

points_detail_final_pivoted_cont_test.display()

In [0]:
# Revised Calculations from Shumaila's code
points_detail_final_pivoted_cont_test = points_detail_final_pivoted_cont_test.withColumn('sales_test_total', f.col('sales_test_total').cast('double'))

points_detail_final_pivoted_output = (
    points_detail_final_pivoted_cont_test
        .dropDuplicates()
        .withColumn('inc_total_points_earned_scaled', (f.col('TEST_measured_hhs_earned') * f.col('scale_factor')))
        .withColumn('cost_inc_total_points_earned', f.round(f.col('inc_total_points_earned_scaled') * 0.01, 2).cast('double'))
        .withColumn('adj_sales_uplift', f.col('sales_uplift_total').cast('double'))
        .withColumn('adj_sales_total', f.col('sales_test_total').cast('Integer'))
        .withColumn('camp_cost', f.col('camp_cost_mmci'))
        .withColumn('working_cost', f.col('working_cost').cast('Integer'))
        .withColumn('adj_total_cost', f.round((f.col('cost_inc_total_points_earned') + f.col('camp_cost')).cast('double'), 2))
        .withColumn('abs_total_cost', f.round((f.col('cost_inc_total_points_earned') + f.col('working_cost')).cast('double'), 2))
        # Added Redemption Cost and Abs Uplift as Adj - Redemption Cost
        .withColumn('adj_sales_uplift', f.col('sales_uplift_total').cast('double'))
        .withColumn('adj_sales_total', f.col('sales_test_total').cast('double'))
        .withColumn('redemption_cost', f.col('abs_total_cost') - f.col('working_cost'))
        .withColumn('new_sales_uplift_earned', f.round((f.col('adj_sales_uplift') - f.col('redemption_cost')).cast('double'), 2))
        .withColumn('new_sales_test_earned', f.round((f.col('adj_sales_total') - f.col('redemption_cost')).cast('double'), 2))
        .withColumn('adj_iroas', f.round((f.col('adj_sales_uplift') / f.col('adj_total_cost')).cast('double'), 2))
        .withColumn('abs_iroas', f.round((f.col('new_sales_uplift_earned') / f.col('abs_total_cost')).cast('double'), 2))
        .withColumn('adj_aroas', f.round((f.col('adj_sales_total') / f.col('adj_total_cost')).cast('double'), 2))
        .withColumn('abs_aroas', f.round((f.col('new_sales_test_earned') / f.col('abs_total_cost')).cast('double'), 2))
        
)

# Keep only adjusted and abs cols
points_detail_final_pivoted_output.display()

In [0]:
# Appending to file rather than overwriting so that we only need to run automation for NEW campaigns (once absolute #s are calculated for a campaign that has finished measurement, those #s dont change)
points_detail_final_pivoted_output.coalesce(1).write.mode("append").parquet('abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/absolute_iroas_xcm_disp')

#final_result_xcm_disp_df_test = spark.read.parquet(f'abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/absolute_iroas_xcm_disp')
#final_result_xcm_disp_df_test.display()

#### TDC, EMOD iROAS automation
- TDC and standalone EMODs handled here 

In [0]:
abs_emod_tdc = spark.read.parquet(f'abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/absolute_iroas_tdc')
abs_emod_tdc.display()

In [0]:
# List of TDCs, dashboard already filters to Top Performer (%800% Product Group) and All Modalities
closed_loop_tdc = closed_loop_prepivot.filter(
    ((f.col("campaign_type") == "TDC") | (f.col("campaign_type") == "EMOD")) &
    (f.year(f.col("camp_start_date")).isin([2024, 2025, 2026]))
)

kpf_dashboard_revamped_tdc = closed_loop_tdc.filter(
    (f.col('campaign_id') != 120767) &
    (~f.col('project_name').contains("XCM"))
)

campaign_id_list = [row['campaign_id'] for row in kpf_dashboard_revamped_tdc.select('campaign_id').distinct().collect()]
kpf_dashboard_revamped_tdc.display()

In [0]:
# Offer data (Project Id, coupon barcode, redemption barcode, effective date, expiration date)
mmoi_metadata = (mmoi
    .join(kpf_dashboard_revamped_tdc.select('kpm_duplicated_id').distinct(), mmoi.KPM_PROJECT_ID == kpf_dashboard_revamped_tdc.kpm_duplicated_id, 'inner')
    .withColumn('EFFECTIVE_DATE', f.date_format(f.to_date('EFFECTIVE_DATE', 'yyyy-MM-dd'), 'yyyyMMdd'))
    .withColumn('EXPIRATION_DATE', f.date_format(f.to_date('EXPIRATION_DATE', 'yyyy-MM-dd'), 'yyyyMMdd'))
    .withColumn('redemption_barcode', f.lpad(f.col('COUPON_BARCODE').cast('string'), 13, '0'))
    .groupBy('KPM_PROJECT_ID')
    .agg(
        f.first('COUPON_BARCODE').alias('coupon_barcode'),
        f.first('redemption_barcode').alias('redemption_barcode'),
        f.first('EFFECTIVE_DATE').alias('effective_date'),
        f.first('EXPIRATION_DATE').alias('expiration_date')
    )
)

# Should be only NEW TDCs that dont exist in dashboard
mmoi_metadata.display()

In [0]:
# Add Working Cost Column. TDC multiplier is 0.141, EMOD multiplier is 0.015
kpf_dashboard_revamped_tdc = (kpf_dashboard_revamped_tdc
    .withColumn(
        'working_cost',
        f.when(f.col('campaign_type') == 'EMOD', (f.col('camp_cost') * 0.015))
         .otherwise(f.col('camp_cost') * 0.141)
         .cast('double')
    )
)

# Join mmci (data already exists in dashboard revamped) with mmoi barcode/date information
kpf_dashboard_revamped_mmoi_tdc = (kpf_dashboard_revamped_tdc
    .join(mmoi_metadata, f.col('kpm_duplicated_id') == f.col('KPM_PROJECT_ID'), how='inner')
    .drop('KPM_PROJECT_ID')
)

kpf_dashboard_revamped_mmoi_tdc.display()

In [0]:
th_filter = (target_history
    .filter(f.col('test_control_id') == '1') # Filter early to reduce volume
    .withColumnRenamed('HSHD_CODE', 'ehhn')
    .join(
        f.broadcast(kpf_dashboard_revamped_mmoi_tdc.select('target_id', 'campaign_id', 'kpm_duplicated_id', 'project_name')),
        on='target_id', 
        how='inner' # This automatically drops any target_ids not in your dashboard
    )
)

th_filter.display()

In [0]:
# Prepare necessary Points Detail SLOW

'''
points_detail = (spark.read.parquet(f'abfss://data@sa8451kemprd.dfs.core.windows.net/pls_points_v2/'))
points_detail_target_history = (broadcast(th_filter)
    .join(points_detail, on='ehhn', how='inner'))

points_detail_target_history_offer_info = (points_detail_target_history
    .withColumn('trn_dt', f.to_date('trn_dt', 'yyyyMMdd'))
    .join(kpf_dashboard_revamped_mmoi_tdc.select('kpm_duplicated_id', 'coupon_barcode', 'redemption_barcode', 'effective_date',	'expiration_date'),
          on='kpm_duplicated_id',
          how='inner')
    .filter(
        (f.col('offer') == f.col('coupon_barcode')) |
        (f.col('offer') == f.col('redemption_barcode'))
    )
)

# Filter offers within effective and expiry date
points_detail_target_history_filtered_offer_info = (points_detail_target_history_offer_info
    .filter(f.col('trn_dt').between('effective_date', 'expiration_date'))
)

# deduplicate rows after join so that points detail isnt double counting
points_detail_target_history_offer_info = points_detail_target_history_offer_info.dropDuplicates()

points_detail_target_history_offer_info.display()
'''

In [0]:
# Prepare necessary Points Detail - Faster Ver

points_detail = (spark.read.parquet(f'abfss://data@sa8451kemprd.dfs.core.windows.net/pls_points_v2/'))
points_detail_target_history = (broadcast(th_filter)
    .join(points_detail, on='ehhn', how='inner'))

# Select only columns needed from mmoi_mhtv_mmci_sse
kpf_dashboard_revamped_mmoi_tdc_subset = (kpf_dashboard_revamped_mmoi_tdc
    .select(
        'campaign_id', 
        'coupon_barcode', 
        'redemption_barcode', 
        'effective_date',
        'expiration_date'
    )
)

# Convert transaction date to datetype 
points_detail_target_history = points_detail_target_history.withColumn(
    'trn_dt', f.to_date('trn_dt', 'yyyyMMdd')
)

# Join points detail + target history with dashboard and metadata (iroas, aroas, barcodes, camp info)
points_detail_target_history_offer_info = (
    points_detail_target_history
    .join(
        f.broadcast(kpf_dashboard_revamped_mmoi_tdc_subset),
        on=points_detail_target_history.kpm_duplicated_id == kpf_dashboard_revamped_mmoi_tdc_subset.campaign_id,
        how="inner"
    )
    .filter(
        (f.col('offer') == f.col('coupon_barcode')) | 
        (f.col('offer') == f.col('redemption_barcode'))
    )
)

points_detail_target_history_offer_info = points_detail_target_history_offer_info.dropDuplicates()
points_detail_target_history_offer_info.display()


In [0]:
# Filter to only transaction date is between effective and expiry date
points_detail_target_history_filtered_offer_info = (
    points_detail_target_history_offer_info
    .filter(
        (f.col('trn_dt') >= f.to_date(f.col('effective_date'), 'yyyyMMdd')) &
        (f.col('trn_dt') <= f.to_date(f.col('expiration_date'), 'yyyyMMdd'))
    )
)

points_detail_target_history_filtered_offer_info.display()

In [0]:
#  Sum of points earned across all HHs per campaign
aggregated_df = (points_detail_target_history_filtered_offer_info
    .groupBy('kpm_duplicated_id', 'project_name')
    .agg(
        f.count('ehhn').alias('total_hhs'), 
        f.approx_count_distinct('ehhn').alias('distinct_ehhn'), 
        f.sum('points_earned').alias('total_points_earned')
    )
)

aggregated_df.display()

In [0]:
# Calculations (from Shumaila's code)
final_result_tdc_df = (
    # Force Spark to broadcast the small table to eliminate the shuffle phase
    f.broadcast(kpf_dashboard_revamped_mmoi_tdc)
    .join(aggregated_df, on=['kpm_duplicated_id', 'project_name'], how='left')
    .withColumn('cost_total_points_earned', f.round(f.col('total_points_earned') * 0.01, 2).cast('double'))
    .withColumn('adj_sales_uplift', f.col("sales_uplift_total").cast('double'))
    .withColumn('adj_sales_total', f.col("sales_test_total").cast('double'))
    .withColumn('camp_cost', f.round(f.col("camp_cost").cast('double'), 2))
    .withColumn('working_cost', f.round(f.col("working_cost").cast('double'), 2))
    .withColumn('adj_total_cost', f.round((f.col('cost_total_points_earned') + f.col('camp_cost')).cast('double'), 2))
    .withColumn('abs_total_cost', f.round((f.col('cost_total_points_earned') + f.col('working_cost')).cast('double'), 2))
    .withColumn('redemption_cost', f.col('abs_total_cost') - f.col('working_cost'))
    .withColumn('abs_sales_uplift_earned', f.round((f.col('adj_sales_uplift') - f.col('redemption_cost')).cast('double'), 2))
    .withColumn('abs_sales_test_earned', f.round((f.col('adj_sales_total') - f.col('redemption_cost')).cast('double'), 2))
    .withColumn('adj_iroas', f.round((f.col('adj_sales_uplift') / f.col('adj_total_cost')).cast('double'), 2))
    .withColumn('abs_iroas', f.round((f.col('abs_sales_uplift_earned') / f.col('abs_total_cost')).cast('double'), 2))
    .withColumn('adj_aroas', f.round((f.col('adj_sales_total') / f.col('adj_total_cost')).cast('double'), 2))
    .withColumn('abs_aroas', f.round((f.col('abs_sales_test_earned') / f.col('abs_total_cost')).cast('double'), 2))
)

# Display the final aggregated DataFrame
final_result_tdc_df.display()

# kpm_duplicated_id, project_name, campaign_id, pg_master, job_id, campaign_type, abs_aroas, adj_aroas, abs_iroas, adj_iroas, abs_total_cost, adj_total_cost, abs_sales_test_earned, abs_sales_uplift_earned, adj_sales_total, adj_sales_uplift, cost_total_points_redeemed, cost_total_points_earned,total_points_earned,

In [0]:
spark.catalog.clearCache()

final_result_tdc_df.select(
    "kpm_duplicated_id", "project_name", "campaign_id", "pg_master", "job_id", "campaign_type",
    "abs_aroas", "adj_aroas", "abs_iroas", "adj_iroas", "abs_total_cost", "adj_total_cost",
    "abs_sales_test_earned", "abs_sales_uplift_earned", "adj_sales_total", "adj_sales_uplift",
    "cost_total_points_earned", "total_points_earned", "working_cost", "redemption_cost"
).coalesce(1).write.mode("overwrite").parquet(
    'abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/absolute_iroas_tdc'
)

#final_result_tdc_df_test = spark.read.parquet('abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/absolute_iroas_tdc')
#final_result_tdc_df_test.display()

#### MCP SSE iROAS automation
- SSE and OL SSE Separate in the future

In [0]:
abs_mcp_sse = spark.read.parquet(f'abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/absolute_iroas_mcp_sse')
abs_mcp_sse.display()

In [0]:
# List of SSEs, dashboard already filters to Top Performer (%800% Product Group) and All Modalities
kpf_dashboard_revamped_sse_pre = closed_loop_prepivot.filter(
  (f.col("campaign_type") == "SSE") &
  (f.year(f.col("camp_start_date")) == 2026)
)

kpf_dashboard_revamped_sse = kpf_dashboard_revamped_sse_pre.join(
    abs_mcp_sse, 
    on=['campaign_id'], 
    how='leftanti'
)

#campaign_id_list = [row['campaign_id'] for row in kpf_dashboard_revamped_sse.select('campaign_id').distinct().collect()]

kpf_dashboard_revamped_sse.display()

In [0]:
# Offer data (Project Id, coupon barcode, redemption barcode, effective date, expiration date)
mmoi_metadata = (mmoi
    .withColumn('EFFECTIVE_DATE', f.date_format(f.to_date('EFFECTIVE_DATE', 'yyyy-MM-dd'), 'yyyyMMdd'))
    .withColumn('EXPIRATION_DATE', f.date_format(f.to_date('EXPIRATION_DATE', 'yyyy-MM-dd'), 'yyyyMMdd'))
    .withColumn('redemption_barcode', f.lpad(f.col('COUPON_BARCODE').cast('string'), 13, '0'))
    .groupBy('KPM_PROJECT_ID')
    .agg(
        #f.first('COUPON_BARCODE').alias('coupon_barcodes'),
        #f.first('redemption_barcode').alias('redemption_barcodes'),
        f.collect_set('COUPON_BARCODE').alias('coupon_barcodes'),
        f.collect_set('redemption_barcode').alias('redemption_barcodes'),
        f.first('EFFECTIVE_DATE').alias('effective_date'),
        f.first('EXPIRATION_DATE').alias('expiration_date')
    )
)

mmoi_metadata.display()

In [0]:
# Calculate working cost, iroas (sales uplift / campaign cost), aroas (sales test total / campaign cost)
mhtv_metrics_agg = (kpf_dashboard_revamped_sse
    .groupBy('campaign_id')
    .agg(
        f.sum('sales_uplift_total').alias('sales_uplift_total'),
        f.sum('sales_test_total').alias('sales_test_total'),
        f.sum('camp_cost').alias('camp_cost')
    )
    .withColumn('working_cost', f.col('camp_cost') * 0.3390)
    .withColumn('iroas_original', f.round(f.col('sales_uplift_total') / f.col('camp_cost'), 2))
    .withColumn('aroas_original', f.round(f.col('sales_test_total') / f.col('camp_cost'), 2))
)

mhtv_metrics_agg.display()

In [0]:
# Join metadata 
mmoi_mhtv_sse = (mmoi_metadata
    .join(mhtv_metrics_agg, mmoi_metadata["KPM_PROJECT_ID"] == mhtv_metrics_agg["campaign_id"], how="right")
)

mmoi_mhtv_mmci_sse = (mmoi_mhtv_sse
    .join(mmci.select('kpm_project_id', 'target_id', 'project_name'), on = "KPM_PROJECT_ID", how = "inner")
)

mmoi_mhtv_mmci_sse.display()

In [0]:
target_ids_to_keep = [
    row['target_id'] for row in mmoi_mhtv_mmci_sse.select('target_id').distinct().collect()
]

# Reduce Target history table with only relevant target ids
test_hhs_slim = (
    target_history
    .filter(f.col('test_control_id') == '1')
    .filter(f.col('target_id').isin(target_ids_to_keep)) 
    .select(f.col('HSHD_CODE').alias('ehhn'), 'target_id')
    .distinct()
)

# Join Target history with metadata
th_filter = (
    test_hhs_slim.join(
        f.broadcast(mmoi_mhtv_mmci_sse.select('kpm_project_id', 'project_name', 'target_id').distinct()), 
        on='target_id', 
        how='inner'
    )
)

th_filter.display()

In [0]:
# Join target history + metadata with points detail
points_detail = (spark.read.parquet(f'abfss://data@sa8451kemprd.dfs.core.windows.net/pls_points_v2/'))
points_detail_target_history = (broadcast(th_filter)
    .join(points_detail, on='ehhn', how='inner'))
display(points_detail_target_history)

In [0]:
# Select only columns needed from mmoi_mhtv_mmci_sse
mmoi_mhtv_mmci_sse_filtered = (mmoi_mhtv_mmci_sse
    .select(
        'kpm_project_id', 
        'coupon_barcodes', 
        'redemption_barcodes', 
        'effective_date',
        'expiration_date'
    )
)

# Convert transaction date to datetype 
points_detail_target_history = points_detail_target_history.withColumn(
    'trn_dt', f.to_date('trn_dt', 'yyyyMMdd')
)

# Join points detail + target history with dashboard and metadata (iroas, aroas, barcodes, camp info)
points_detail_target_history_offer_info = (
    points_detail_target_history
    .join(
        f.broadcast(mmoi_mhtv_mmci_sse_filtered),
        on=[
            points_detail_target_history.kpm_project_id == mmoi_mhtv_mmci_sse_filtered.kpm_project_id,
            (f.array_contains(mmoi_mhtv_mmci_sse_filtered.coupon_barcodes, points_detail_target_history.offer) | 
             f.array_contains(mmoi_mhtv_mmci_sse_filtered.redemption_barcodes, points_detail_target_history.offer))
            #points_detail_target_history.trn_dt.between(mmoi_mhtv_mmci_sse_filtered.effective_date, mmoi_mhtv_mmci_sse_filtered.expiration_date)
        ],
        how="inner"
    ).drop(mmoi_mhtv_mmci_sse_filtered.kpm_project_id)
)

points_detail_target_history_offer_info.display()

In [0]:
# Filter to only transaction date is between effective and expiry date
points_detail_target_history_filtered_offer_info = (points_detail_target_history_offer_info
    .filter(f.col('trn_dt').between('effective_date', 'expiration_date'))
)

points_detail_target_history_offer_info.display()

In [0]:
#  Sum of points earned across all HHs per campaign
aggregated_df = (points_detail_target_history_offer_info
    .groupBy('kpm_project_id', 'project_name')
    .agg(
        f.count('ehhn').alias('total_hhs'), 
        f.countDistinct('ehhn').alias('distinct_ehhn'), 
        f.sum('points_earned').alias('total_points_earned')
    )
)

aggregated_df.display()

In [0]:
# Calculations (from Shumaila's code)
final_result_sse_df = (
    mmoi_mhtv_mmci_sse
    .join(aggregated_df, on=['kpm_project_id', 'project_name'], how='left')
    .withColumn('cost_total_points_earned', f.round(f.col('total_points_earned') * 0.01, 2).cast('double'))
    .withColumn('cost_total_points_redemeed', f.round(f.col('total_points_earned') * 0.016 * 0.5887, 2).cast('double'))
    .withColumn('adj_sales_uplift', f.col("sales_uplift_total").cast('Integer'))
    .withColumn('adj_sales_total', f.col("sales_test_total").cast('Integer'))
    .withColumn('camp_cost', f.round(f.col("camp_cost").cast('double'), 2))
    .withColumn('working_cost', f.round(f.col("working_cost").cast('double'), 2))
    .withColumn('adj_total_cost', f.round((f.col('cost_total_points_earned') + f.col('camp_cost')).cast('double'), 2))
    .withColumn('abs_total_cost', f.round((f.col('cost_total_points_earned') + f.col('working_cost')).cast('double'), 2))
    # Adj Numbers - Redemption Cost to better match Shumaila's Numbers
    .withColumn('redemption_cost', f.col('abs_total_cost') - f.col('working_cost'))
    .withColumn('abs_sales_uplift_earned', f.round((f.col('adj_sales_uplift') - f.col('redemption_cost')).cast('double'), 2))
    .withColumn('abs_sales_test_earned', f.round((f.col('adj_sales_total') - f.col('redemption_cost')).cast('double'), 2))
    .withColumn('adj_iroas', f.round((f.col('adj_sales_uplift') / f.col('adj_total_cost')).cast('double'), 2))
    .withColumn('abs_iroas', f.round((f.col('abs_sales_uplift_earned') / f.col('abs_total_cost')).cast('double'), 2))
    .withColumn('adj_aroas', f.round((f.col('adj_sales_total') / f.col('adj_total_cost')).cast('double'), 2))
    .withColumn('abs_aroas', f.round((f.col('abs_sales_test_earned') / f.col('abs_total_cost')).cast('double'), 2))
)

# Display the final aggregated DataFrame
final_result_sse_df.display()

- QA'd all 2025 Q2 MCP SSEs
- QA'd all 2025 Q3 MCP SSEs


In [0]:
# Dropping array columns (barcodes) to support csv and parquet writing
final_result_sse_df_barcodes_dropped = final_result_sse_df.drop('coupon_barcodes', 'redemption_barcodes')
final_result_sse_df_barcodes_dropped.coalesce(1).write.mode("overwrite").parquet('abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/absolute_iroas_mcp_sse')

#final_result_sse_df_test = spark.read.parquet(f'abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/absolute_iroas_mcp_sse')
#final_result_sse_df_test.display()

#### Processing Final Dataframe 

In [0]:
# Processing MCP SSE campaigns + automation
final_result_sse_df_test = spark.read.parquet(f'abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/absolute_iroas_mcp_sse') \
    .dropDuplicates() \
    .filter(f.col("effective_date") >= "20240101") \
    .withColumnRenamed("abs_sales_uplift_earned", "abs_sales_uplift") \
    .withColumnRenamed("KPM_PROJECT_ID", "kpm_duplicated_id") \
    .withColumnRenamed("redemption_cost", "new_redemption_cost") \
    .select(
        "kpm_duplicated_id", "working_cost", "new_redemption_cost", "adj_sales_uplift", "abs_sales_uplift", "abs_sales_test_earned", "adj_iroas", "abs_iroas", "adj_aroas", "abs_aroas", "adj_total_cost", "abs_total_cost"
    )

# Make negative #s to 0, since we do not report negative #s.
cols_to_zero_neg = [
    "working_cost", "adj_sales_uplift", "abs_sales_uplift", "adj_iroas", "abs_iroas", "adj_aroas", "abs_aroas", "adj_total_cost", "abs_total_cost", "abs_sales_test_earned"
]

for c in cols_to_zero_neg:
    final_result_sse_df_test = final_result_sse_df_test.withColumn(c, f.when(f.col(c) < 0, 0).otherwise(f.col(c)))

final_result_sse_df_test.display()

In [0]:
# Processing REM SSE PUSH XCM campaigns + automation
final_result_rem_sse_push_xcm = spark.read.parquet(f'abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/absolute_iroas_rem_sse_push_xcm') \
    .dropDuplicates() \
    .filter(f.col("effective_date") >= "20240101") \
    .withColumnRenamed("abs_sales_uplift_earned", "abs_sales_uplift") \
    .withColumnRenamed("KPM_PROJECT_ID", "kpm_duplicated_id") \
    .withColumnRenamed("redemption_cost", "new_redemption_cost") \
    .select(
        "kpm_duplicated_id", "working_cost", "new_redemption_cost", "adj_sales_uplift", "abs_sales_uplift", "abs_sales_test_earned", "adj_iroas", "abs_iroas", "adj_aroas", "abs_aroas", "adj_total_cost", "abs_total_cost"
    )

# Make negative #s to 0, since we do not report negative #s.
cols_to_zero_neg = [
    "working_cost", "adj_sales_uplift", "abs_sales_uplift", "adj_iroas", "abs_iroas", "adj_aroas", "abs_aroas", "adj_total_cost", "abs_total_cost", "abs_sales_test_earned"
]

for c in cols_to_zero_neg:
    final_result_rem_sse_push_xcm = final_result_rem_sse_push_xcm.withColumn(c, f.when(f.col(c) < 0, 0).otherwise(f.col(c)))
final_result_rem_sse_push_xcm.display()



In [0]:
# Processing XCM and DISP standalone campaigns + automation
final_result_xcm_disp_df_test = spark.read.parquet(f'abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/absolute_iroas_xcm_disp') \
    .dropDuplicates() \
    .withColumnRenamed("new_sales_uplift_earned", "abs_sales_uplift") \
    .withColumnRenamed("new_sales_test_earned", "abs_sales_test_earned") \
    .withColumnRenamed("redemption_cost", "new_redemption_cost") \
    .select(
        "kpm_duplicated_id", "working_cost", "new_redemption_cost", "adj_sales_uplift", "abs_sales_uplift", "abs_sales_test_earned", "adj_iroas", "abs_iroas", "adj_aroas", "abs_aroas", "adj_total_cost", "abs_total_cost"
    )

# Overwrite values for kpm_duplicated_id == 136421 (Hard coding this campaign since it has no barcodes for now, fix later)
final_result_xcm_disp_df_test = final_result_xcm_disp_df_test.withColumn(
    "abs_sales_uplift",
    f.when(f.col("kpm_duplicated_id") == 136421, f.lit(2813986.5)).otherwise(f.col("abs_sales_uplift"))
).withColumn(
    "adj_iroas",
    f.when(f.col("kpm_duplicated_id") == 136421, f.lit(16.1)).otherwise(f.col("adj_iroas"))
).withColumn(
    "abs_iroas",
    f.when(f.col("kpm_duplicated_id") == 136421, f.lit(30.4)).otherwise(f.col("abs_iroas"))
).withColumn(
    "adj_aroas",
    f.when(f.col("kpm_duplicated_id") == 136421, f.lit(85.0)).otherwise(f.col("adj_aroas"))
).withColumn(
    "abs_aroas",
    f.when(f.col("kpm_duplicated_id") == 136421, f.lit(161.0)).otherwise(f.col("abs_aroas"))
).withColumn(
    "abs_total_cost",
    f.when(f.col("kpm_duplicated_id") == 136421, f.lit(92700)).otherwise(f.col("abs_total_cost"))
).withColumn(
    "adj_total_cost",
    f.when(f.col("kpm_duplicated_id") == 136421, f.lit(175000)).otherwise(f.col("adj_total_cost"))
).withColumn(
    "new_redemption_cost",
    f.when(f.col("kpm_duplicated_id") == 136421, f.lit(0)).otherwise(f.col("new_redemption_cost"))
).withColumn(
    "abs_sales_test_earned",
    f.when(f.col("kpm_duplicated_id") == 136421, f.lit(14913202)).otherwise(f.col("abs_sales_test_earned"))
)

# Overwrite values for kpm_duplicated_id == 147400 (Hard coding this campaign since it has no barcodes for now, fix later)
final_result_xcm_disp_df_test = final_result_xcm_disp_df_test.withColumn(
    "abs_sales_uplift",
    f.when(f.col("kpm_duplicated_id") == 147400, f.lit(1718853.8)).otherwise(f.col("abs_sales_uplift"))
).withColumn(
    "adj_iroas",
    f.when(f.col("kpm_duplicated_id") == 147400, f.lit(9.0)).otherwise(f.col("adj_iroas"))
).withColumn(
    "abs_iroas",
    f.when(f.col("kpm_duplicated_id") == 147400, f.lit(17.5)).otherwise(f.col("abs_iroas"))
).withColumn(
    "adj_aroas",
    f.when(f.col("kpm_duplicated_id") == 147400, f.lit(109.0)).otherwise(f.col("adj_aroas"))
).withColumn(
    "abs_aroas",
    f.when(f.col("kpm_duplicated_id") == 147400, f.lit(210.0)).otherwise(f.col("abs_aroas"))
).withColumn(
    "abs_total_cost",
    f.when(f.col("kpm_duplicated_id") == 147400, f.lit(97979)).otherwise(f.col("abs_total_cost"))
).withColumn(
    "adj_total_cost",
    f.when(f.col("kpm_duplicated_id") == 147400, f.lit(190000)).otherwise(f.col("adj_total_cost"))
).withColumn(
    "new_redemption_cost",
    f.when(f.col("kpm_duplicated_id") == 147400, f.lit(0)).otherwise(f.col("new_redemption_cost"))
).withColumn(
    "abs_sales_test_earned",
    f.when(f.col("kpm_duplicated_id") == 147400, f.lit(20616986)).otherwise(f.col("abs_sales_test_earned"))
)

# Make negative #s to 0, since we do not report negative #s.
cols_to_zero_neg = [
    "working_cost", "adj_sales_uplift", "abs_sales_uplift", "adj_iroas", "abs_iroas", "adj_aroas", "abs_aroas", "adj_total_cost", "abs_total_cost", "abs_sales_test_earned"
]

for c in cols_to_zero_neg:
    final_result_xcm_disp_df_test = final_result_xcm_disp_df_test.withColumn(c, f.when(f.col(c) < 0, 0).otherwise(f.col(c)))

# Remove row where kpm_duplicated_id == 131511 and abs_iroas == 0
final_result_xcm_disp_df_test = final_result_xcm_disp_df_test.filter(~((f.col("kpm_duplicated_id") == 131511) & (f.col("abs_iroas") == 0)))

#136421, 147400
final_result_xcm_disp_df_test.display()

In [0]:
# Processing TDC campaigns + automation
final_result_tdc_df_test = spark.read.parquet(
  f'abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/absolute_iroas_tdc'
).dropDuplicates() \
 .withColumnRenamed("abs_sales_uplift_earned", "abs_sales_uplift") \
 .withColumnRenamed("redemption_cost", "new_redemption_cost") \
 .withColumn("adj_sales_total", f.col("adj_sales_total").cast("double")) \
 .select(
    "kpm_duplicated_id", "working_cost", "new_redemption_cost", "adj_sales_uplift", "abs_sales_uplift", "abs_sales_test_earned", "adj_iroas", "abs_iroas", "adj_aroas", "abs_aroas", "adj_total_cost", "abs_total_cost"
)

# Make negative #s to 0, since we do not report negative #s.
cols_to_zero_neg = [
    "working_cost", "adj_sales_uplift", "abs_sales_uplift", "adj_iroas", "abs_iroas", "adj_aroas", "abs_aroas", "adj_total_cost", "abs_total_cost", "abs_sales_test_earned"
]

for c in cols_to_zero_neg:
    final_result_tdc_df_test = final_result_tdc_df_test.withColumn(c, f.when(f.col(c) < 0, 0).otherwise(f.col(c)))

final_result_tdc_df_test.display()

In [0]:
# Combine TDC, XCM, SSE ABS numbers into one dataframe and write it out
tdc_sse_xcm_disp_abs_df = final_result_tdc_df_test.union(final_result_xcm_disp_df_test).union(
  final_result_sse_df_test).union(final_result_rem_sse_push_xcm)
tdc_sse_xcm_disp_abs_df.display()

tdc_sse_xcm_disp_abs_df.coalesce(1).write.mode("overwrite").parquet('abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/absolute_iroas_all_camp_types')